# Previsão de Churn para Clientes do Banco
## Bank Customer Churn
"Your task is to predict whether a customer continues with their account or closes it."
kaggle.

Dataset original:
https://www.kaggle.com/competitions/bank-customer-churn-ict-u-ai/overview



#Etapas do Projeto:

**1- O problema de negócio:** objetivo principal do projeto e como ele gera valor ao banco.

**2- Coleta e entendimento dos dados:** subir no github e validar importação.

**3- Dividir os dados em conjuntos de treino e teste:** Separar uma parte dos dados para ensinar o algoritmo e outra parte isolada para validar sua capacidade de generalização.

**4- Análise exploratória de dados - EDA:** Identificar padrões visuais, distribuições, outliers e entender como as variáveis se relacionam com o alvo.

**5- Engenharia de features, limpeza de dados e pré-processamento:** Tratar valores nulos, codificar categorias e criar novas variáveis calculadas para facilitar o aprendizado do algoritmo.

**6- Treinamento do modelo, comparação, seleção de features e ajuste (tuning):** Testar diferentes algoritmos, selecionar as colunas mais importantes e otimizar os hiperparâmetros para obter a melhor performance.

**7- Teste e Avaliação Final do Modelo** (Hold-out Validation)

**8 - Concluir e interpretar os resultados do modelo:** Explicar as decisões do algoritmo (usando técnicas como valores SHAP) para entender o que impulsiona o comportamento mapeado.

**9- Implantação (Deploy):** Colocar o modelo em produção via API ou web app (ou, no contexto de uma competição, gerar e submeter o arquivo final de predições).


## Observações:
Essas Etapas foram definidas, tomando como base a métodologia CRISP-DM.

Esse Notebook contém a execução das 4 primeiras etapas, responsáveis pela EDA do projeto. As analises feitas aqui vizam observar padrões e entender o perfil dos clientes que geram Churns. Essa documentação é fundamental para a melhora do modelo preditivo e comprovação real e estatistica dos resultados de saída.



In [12]:
import os
import sys
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# Definir o caminho base do projeto
project_path = '/content/drive/MyDrive/Colab_Notebooks/Machine_Learning/Bank_Churn_Model'

# Adicionar funcoes do Scripts
# vizualizacao.py
sys.path.append(os.path.join(project_path, 'Scripts'))
from EDA.visualizacao import plotar_distribuicao_numerica
from EDA.visualizacao import plotar_deteccao_outliers
from EDA.visualizacao import plotar_distribuicao_categorica
from EDA.visualizacao import plotar_diagnostico_correlacao


In [17]:
# IMPORTANDO BIBLIOTECAS

# Dados e Vizualizações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import math

# Divisão dos Dados
from sklearn.model_selection import train_test_split


# Etapa 1: O problema de negócio: objetivo principal do projeto e como ele gera valor ao banco.

A diretoria do banco identificou uma taxa preocupante de clientes encerrando suas contas e abandonando os serviços da instituição. O objetivo principal é **prever a probabilidade de um cliente dar churn**, permitindo que o banco aja de forma proativa, oferecendo serviços melhores e revertendo a decisão de cancelamento antes que ela ocorra.

### 1.1 Contexto e KPIs Financeiros
Quando um banco adquire um novo cliente, três Indicadores-Chave de Desempenho (KPIs) fundamentais entram em jogo:

* **Custo de Aquisição de Clientes (CAC):** Mede as despesas associadas à atração de cada novo cliente, englobando marketing, vendas e custos operacionais. Um CAC menor reflete eficiência na aquisição.
* **Valor do Tempo de Vida do Cliente (CLV - Customer Lifetime Value):** Estima a receita total que o banco espera gerar com o cliente ao longo de todo o relacionamento. Um CLV alto indica que o valor do cliente supera o custo de aquisição, garantindo lucratividade a longo prazo.
* **Taxa de Evasão (Churn Rate):** Representa a porcentagem de clientes que deixaram o banco durante um período específico.

Como sabemos, é muito mais caro adquirir um novo cliente do que reter um atual. Para maximizar a lucratividade, o banco precisa **minimizar o CAC e o Churn**, enquanto **maximiza o CLV**.

### 1.2 Objetivos do Projeto
1. **Identificar os fatores críticos** associados à evasão de clientes bancários.
2. **Construir um modelo preditivo robusto** capaz de calcular com precisão a probabilidade de churn de cada cliente.
3. **Fornecer insights acionáveis** para que o banco desenvolva planos de retenção eficientes.

### 1.3 Por que prever probabilidades (Score Ordering) em vez de classes binárias?
Ao colocar o modelo em produção, o objetivo não é simplesmente dizer se o cliente vai cancelar (`1`) ou não (`0`). O valor real para o negócio está em **gerar um score de probabilidade** para cada usuário.

Essa abordagem (diretamente ligada à métrica ROC-AUC) permite uma tomada de decisão inteligente e alocação eficiente de recursos. O banco pode ordenar a base de clientes do mais propenso ao menos propenso ao churn. Com isso, os esforços da equipe de retenção e o orçamento de marketing são direcionados cirurgicamente para os clientes de alto risco e alto valor, otimizando o retorno sobre o investimento (ROI).

### 1.4 Benefícios Esperados
* **Redução de Custos:** Evita gastos desnecessários com campanhas em massa.
* **Retenção Inteligente:** Foco nos clientes que realmente importam e estão em risco.
* **Proteção de Receita:** Manutenção do capital na instituição.
* **Melhoria na Experiência do Cliente:** Identificação e correção de atritos de forma proativa.

# Etapa 2: Coleta e entendimento dos dados:

In [ ]:
from google.colab import userdata

# Recuperando as chaves dos segredos do Colab
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USER')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Baixar o dataset da competição
!kaggle competitions download -c bank-customer-churn-ict-u-ai

# Descompactar
!unzip -q bank-customer-churn-ict-u-ai.zip -d data_churn

401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
unzip:  cannot find or open bank-customer-churn-ict-u-ai.zip, bank-customer-churn-ict-u-ai.zip.zip or bank-customer-churn-ict-u-ai.zip.ZIP.


In [ ]:
# Tente baixar novamente após aceitar as regras no site
!kaggle competitions download -c bank-customer-churn-ict-u-ai -p /content

# Verifique o que foi baixado antes de descompactar
!ls /content/*.zip

401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
ls: cannot access '/content/*.zip': No such file or directory


In [ ]:
# URLs (raw) do repositório no GitHub
# Dados de treino para o modelo
url_train = "https://raw.githubusercontent.com/GabrielSenkovski/Bank_Churn_Model/refs/heads/main/Datasets/train.csv"

# DataFrame
df_dados = pd.read_csv(url_train)

# Imprimindo shape
print(f"Dataset de Treino carregado com sucesso: {df_dados.shape[0]} linhas e {df_dados.shape[1]} colunas.")

# Visualizando as primeiras linhas do conjunto de treino
display(df_dados.head())

HTTPError: HTTP Error 404: Not Found

###  Dicionário de Dados e Contexto de Negócio

**Objetivo e Contexto:**
No mercado financeiro, é muito mais caro adquirir um novo cliente (Custo de Aquisição de Cliente - CAC) do que reter um já existente. Por isso, é altamente vantajoso para o banco entender os fatores que levam um cliente a encerrar sua conta. A prevenção da evasão (*churn*) permite o desenvolvimento de campanhas de retenção e programas de fidelidade bem direcionados.

**Descrição das Variáveis Preditivas (Features):**

* **RowNumber**: Corresponde ao índice do registro. Não tem efeito preditivo.
* **CustomerId**: Identificador único do sistema. Contém valores aleatórios e não afeta a decisão de saída.
* **Surname**: O sobrenome do cliente. Não tem impacto analítico na decisão de deixar o banco.
* **CreditScore**: Pode influenciar o *churn*, pois clientes com pontuações de crédito mais altas geralmente têm menos probabilidade de sair.
* **Geography**: A localização do cliente, um fator regional que pode afetar a taxa de evasão.
* **Gender**: Permite explorar se homens ou mulheres têm taxas de saída diferentes.
* **Age**: Variável de alta relevância; historicamente, clientes mais velhos são menos propensos a deixar o banco do que os mais jovens.
* **Tenure**: Número de anos que a pessoa é cliente. Clientes mais antigos costumam ser mais fiéis.
* **Balance**: Excelente indicador. Clientes com saldos maiores em conta têm menos probabilidade de sair em comparação com aqueles com saldos zerados ou baixos.
* **NumOfProducts**: Quantidade de serviços do banco utilizados pelo cliente (ex: conta, empréstimo, investimentos).
* **HasCrCard**: Flag binária. Relevante, pois a posse de um cartão de crédito tende a aumentar a retenção.
* **IsActiveMember**: Indica se o cliente movimenta a conta frequentemente. Clientes ativos têm menor probabilidade de *churn*.
* **EstimatedSalary**: Similar ao saldo bancário, clientes com salários menores tendem a ser mais voláteis do que clientes com altas rendas.
* **Complain**: Indica se o cliente já registrou alguma reclamação no atendimento.
* **Satisfaction Score**: Pontuação dada pelo cliente avaliando a resolução de sua reclamação.
* **Card Type**: A categoria do cartão de crédito do cliente (ex: Gold, Platinum, etc.).
* **Points Earned**: Quantidade de pontos de recompensa ganhos pelo cliente ao usar o cartão de crédito.

**Variável Alvo (Target):**

* **Exited** (Churn): O resultado que queremos prever. Indica se o cliente deixou o banco (1 = Sim / 0 = Não).

In [ ]:
#Verificando o tipo de dados, numero de instancias e se aceita Nulos.
df_dados.info()

# complementando, imprimindo o numero de instancias e classes
print(f'\n O data set {df_dados.shape[0]} linhas e {df_dados.shape[1]} colunas.')

In [ ]:
# Verificando se há valores nulos
df_dados.isna().sum()


In [ ]:
# Verificando se há valores duplicados
print(f'\n O data set possui {df_dados.duplicated().sum()} linhas duplicadas.')

Imprimindo a distribuição dos quartis e média e outras métricas do dataframe usando o describe

In [ ]:
df_dados.describe().T

## O que podemos observar aqui?
### Há algumas observações sobre o perfil geral dos clientes que podemos perceber:


*   A idade média é de aproximadamente 38 anos. Metade dos clientes tem entre 32 e 42 anos, indicando um perfil de adultos jovens a de meia-idade.

* Em média, os clientes possuem aproximadamente 1.6 produtos do banco. A grande maioria (pelo menos 75%) possui 1 ou 2 produtos, e ninguém possui mais que 4.

* Pelo menos 50% dos clientes têm saldo zero em suas contas (a mediana é 0.0). **Este é um indicador crítico de negócio**. O banco precisa investigar por que metade de sua base não está mantendo dinheiro na conta, já que isso limita a capacidade do banco de gerar receita através desses fundos.  
  
* Menos da metade dos clientes (48,6%) são membros ativos. **Isso é um mau indicador**. O banco deve planejar estratégias para melhorar o engajamento dos clientes, já que a baixa atividade frequentemente antecede o cancelamento.

* Aproximadamente 78,5% dos clientes possuem um cartão de crédito, demonstrando uma forte penetração deste produto específico na base.

* O score de crédito médio é 658. Metade dos clientes possui uma pontuação entre 603 e 709, representando um perfil de crédito geralmente estável e saudável.

* A taxa geral de evasão (variável alvo Exited) é de aproximadamente 20,5%. **Isso significa que 1 em cada 5 clientes está deixando o banco**, uma alta taxa de atrito que justifica o esforço de modelagem preditiva.

* Além disso, observando os valores mínimos e máximos (ex: idade mínima de 18 anos, tempo de relacionamento de 0), parece que não há valores inconsistentes ou erros de digitação nestas variáveis.

## Removendo Colunas Desnecessárias

In [ ]:
# Criando uma cópia para preservar o dado original em caso de erro
df_eda = df_dados.copy()

# Removendo variáveis de identificação que não geram valor preditivo
colunas_para_dropar = ['id', 'CustomerId', 'Surname']
df_eda.drop(columns=colunas_para_dropar, inplace=True)
# Verifica Colunas Restantes
print(f"Colunas restantes para análise: {df_eda.columns.tolist()}")


## Declarando Tipos de Variáveis: Numéricas e Categóricas

In [ ]:
# DEFININDO VARIÁVEIS NUMÉRICAS
cols_num = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary']

# DEFININDO VARIÁVEIS CATEGÓRICAS
cols_cat = ['Geography', 'Gender', 'HasCrCard', 'IsActiveMember', 'NumOfProducts']

# Validação de sucesso
print(f"Features Numéricas: {len(cols_num)}")
print(f"Features Categóricas: {len(cols_cat)}")

# Etapa 3: Dividir os dados em conjuntos de treino e teste (SPIT)


Antes de qualquer análise, o primeiro passo é dividir os dados em conjuntos de **treino** e **teste**.

* **Prevenção de Vazamento de Dados (*Data Leakage*):** O conjunto de teste simula dados do mundo real que o algoritmo nunca viu. Para garantir uma avaliação correta e realista do modelo no futuro, toda a Análise Exploratória de Dados (EDA) será realizada **exclusivamente no conjunto de treinamento**.

* **Estratificação (stratify=y)**: Como o dataset é desbalanceado (onde o número de "churns" é muito menor que não "churns"), o parâmetro "stratify=y" sera usado na função de divisão do DF. Isso garante que a mesma proporção percentual da variável alvo seja mantida tanto no treino quanto no teste, preservando a representatividade do cenário real.

In [ ]:
# Definindo X e y
X = df_eda.drop(columns='Exited')
y = df_eda['Exited']

# Definindo Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
# Verificando numero linhas e colunas
print("Tamanho de X_train:", X_train.shape)
print("Tamanho de X_test:", X_test.shape, "\n")

# Testando a estratificação (a porcentagem de 0s e 1s deve ser idêntica nos dois grupos)
print("Proporção de Churn no Treino:")
print(y_train.value_counts(normalize=True) * 100)

print("\nProporção de Churn no Teste:")
print(y_test.value_counts(normalize=True) * 100)


Observa-se que há um desbalanceamento severo na coluna Target (Exited), o que deve ser considerado na hora de treinar o modelo. Esse cenário afeta diretamente métricas como a precisão, podendo enviesar o algoritmo a prever apenas a classe majoritária. Logo, exigirá soluções específicas na etapa de modelagem, como o ajuste de pesos das classes e a escolha de métricas mais robustas, como a ROC-AUC.

# Etapa 4: Análise exploratória de dados - EDA
## 4.1 Dados Numéricos:

In [ ]:
# Juntando as features (X_train) com o alvo (y_train) lado a lado
# Isso é necessário para fazer as analises comparativas com o Target
df_train = pd.concat([X_train, y_train], axis=1)

# Verificando se deu certo
print("Tamanho do df_train:", df_train.shape)
display(df_train.head()) # deve mostrar as colunas de X mais a coluna 'exited' no final



In [ ]:
# Chama a função para plotar os gráficos das variáveis numéricas
plotar_distribuicao_numerica(df_train, cols_num)

## Diagnóstico Numérico e Fatores Críticos de Churn

### 1. Fuga de Capital (*Capital Leakage*)
* **Risco Crítico:** Clientes com saldos elevados (entre **100k e 150k**) apresentam a maior taxa de evasão.
* **Paradoxo da Retenção:** Clientes com saldo zerado são proporcionalmente os mais estáveis.
* **Impacto:** O banco sofre com a perda de AUM (*Assets Under Management*), mantendo contas de baixo valor residual e perdendo capital rentável.

### 2. Vulnerabilidade Geracional
* **Ponto de Alerta:** A faixa de **45 a 55 anos** concentra o risco máximo de saída.
* **Ponto Seguro:** O público jovem (<35 anos) é a âncora de estabilidade da instituição.
* **Hipótese:** O portfólio de produtos perde competitividade (em taxas ou atratividade) exatamente quando o cliente atinge a maturidade financeira.

### 3. Inércia de Relacionamento e Renda
* **Fatores Neutros:** Tempo de relacionamento (Tenure) e Salário Estimado (EstimatedSalary) mostraram-se estatisticamente homogeneos para prever a saída.
* **Insight:** O tempo de relacionamento não retém clientes, e a renda não prevê a evasão. O fator decisivo é o capital (saldo) efetivamente depositado na instituição.

**Conclusão da Análise:** O banco enfrenta um problema de **atratividade para o segmento Premium**. A retenção atual acontece por conveniência (baixo saldo), enquanto a perda acontece por performance (alto saldo). A estratégia de negócios precisa ser redirecionada urgentemente para blindar o público de meia-idade com alto patrimônio agregado.

### Verificação de OutLiers

In [ ]:
plotar_deteccao_outliers(df_train, cols_num)

In [ ]:
# Filtrando o DataFrame para manter apenas quem ganha menos de 1 milhão
df_train = df_train[df_train['EstimatedSalary'] < 1000000]

# Verificando se a linha foi removida
print(f"Novo tamanho do dataset: {df_train.shape[0]} linhas")

A análise via Boxplots confirmou que os valores atípicos em Age, CreditScore e Balance são condizentes com a variabilidade real do negócio e serão mantidos.


Identificou-se uma única instância ruidosa em EstimatedSalary (> 1M), discrepante da distribuição geral (limite de 200k), que será removida para evitar distorções estatísticas no modelo.


## 4.2 Dados Categóricos:
Proporção de Churn em cada Grupo.


In [ ]:
# Chama a função para plotar os gráficos das variáveis categóricas
plotar_distribuicao_categorica(df_train, cols_cat)

## Diagnóstico Categórico e Perfis de Risco

### 1. A "Armadilha" do Cross-Selling (Sobre Vendas)
* **Ponto Ideal (*Sweet Spot*):** Clientes com exatos **2 produtos** apresentam a maior taxa de fidelização.
* **Risco Crítico:** A posse de **3 ou 4 produtos** eleva a evasão para quase 100%.
* **Impacto:** Estratégias de venda agressiva ou excesso de complexidade geram atrito, motivando o cliente a encerrar o relacionamento.

### 2. Fissuras Regionais e de Gênero
* **Geografia:** A operação na **Alemanha** apresenta evasão drasticamente superior à da França e Espanha.
* **Gênero:** O público feminino possui uma propensão significativamente maior ao churn.
* **Hipótese:** Déficit de competitividade na oferta alemã (concorrência forte ou taxas ruins) e um portfólio que não atende plenamente às necessidades financeiras femininas.

### 3. Engajamento Real vs. Posse de Produtos
* **Sinal de Alerta:** A inatividade da conta (IsActiveMember) = 0 é o preditor categórico mais forte de saída.
* **Elemento Neutro:** A posse de um cartão de crédito (HasCrCard) não está retendo os clientes no banco.
* **Insight:** Fidelização exige uso contínuo por parte do cliente. Distribuir "produtos de prateleira" (como cartões não utilizados) não evita o churn.

> **Conclusão da Análise:** O perfil de **Risco Máximo** é o cliente **Alemão, Inativo e com 3+ produtos**. A instituição deve suspender políticas de empurrar múltiplos produtos sem garantir o uso diário da conta. A estratégia deve migrar imediatamente de *"vender mais"* para *"engajar melhor os produtos"*.

### Investigando Hipotestes dos Diagnósticos Noméricos e Categóricos

In [ ]:
# Calcula a taxa média de evasão (churn rate) para o cruzamento das duas variáveis
prova_matematica = df_train.groupby(['Geography', 'IsActiveMember'])['Exited'].mean().reset_index()

# Converte para porcentagem para facilitar a leitura
prova_matematica['Taxa_de_Churn_%'] = (prova_matematica['Exited'] * 100).round(2)

# Exibe os resultados ordenados pelo maior risco
prova_matematica = prova_matematica.sort_values(by='Taxa_de_Churn_%', ascending=False)
display(prova_matematica[['Geography', 'IsActiveMember', 'Taxa_de_Churn_%']])

In [ ]:


plt.figure(figsize=(10, 6))

# O barplot com y='Exited' calcula automaticamente a média (taxa de churn)
sns.barplot(
    x='Geography',
    y='Exited',
    hue='IsActiveMember',
    data=df_train,
    errorbar=None,
    palette='Set2'
)

plt.title('Taxa de Churn: País vs Status de Atividade', fontsize=14, pad=15)
plt.ylabel('Taxa de Churn (Proporção)', fontsize=12)
plt.xlabel('País', fontsize=12)
plt.show()

## Diagnóstico de Correlações (Heat Map)


In [ ]:
corr_matrix = df_train.corr(numeric_only=True)

corr_matrix['Exited'].sort_values(ascending=False)

### Diagnóstico de Correlação e Multicolinearidade

Para garantir a robustez do modelo de Machine Learning, precisamos responder a duas perguntas fundamentais:

**1. Quais variáveis influenciam linearmente a saída do cliente?**
Avaliando a correlação com o *Target* (Exited), notamos que relações estritamente lineares são fracas. Isso indica que o churn é um comportamento complexo e não-linear, exigindo modelos baseados em árvores (como XGBoost ou Random Forest).

**2. Nossos dados possuem informações matemáticas duplicadas?**
Abaixo, plotamos a matriz de multicolinearidade. O objetivo é garantir que não existam variáveis independentes altamente correlacionadas entre si (acima de 0.80), o que causaria redundância e prejudicaria o aprendizado do algoritmo.

In [ ]:
# Chama a função de mapa de Correlações
plotar_diagnostico_correlacao(df_train, cols_num)

# Featuare POOL - criação de variaveis potencias baseadas na minha analise grafica
exemplo: "is_german_inative", "is_risk_zone", etc..
esse conjunto de variaveis deve ser testado nesse caso com Information Value,
dada a natureza favoravel aos modelos de arvore, pois analisa o impacto da var em prever o churn, de forma univariada.

EX; Fetuare Pool = X + (variaveis criadas)

o Teste de  IV deve ser aplicado sobre o fetuare pool


A Poda (Contração): Você aplica a regra de corte (IV < 0.02).
Deve-se descartar todas as variaveis que não atingir o IV < 0.02
(Muito perigoso perder combinações onde a junção de duas Var "ruins" seriam boas


Utilizando a Matriz de Correlação, buscamos validar duas premissas cruciais para a modelagem:

* **Quais variáveis influenciam linearmente a evasão?**
Observando a variável alvo (Exited), notamos que apenas (Age) apresenta um sinal moderado (0.46). A baixa correlação linear geral reforça que o churn é um problema de comportamento não linear, exigindo algoritmos baseados em árvores (como Random Forest, XGBoost ou CatBoost).

* **Existe redundância matemática nos dados?**
Avaliando o cruzamento interno entre as variáveis preditoras, não identificamos nenhum caso de multicolinearidade (correlações fortes, > 0.80). Os dados estão limpos de redundância e seguros para a modelagem.

# Conclusão da EDA e Escolha do Modelo Base

**O Problema Central:** O banco sofre uma hemorragia seletiva de capital, perdendo seus clientes financeiramente mais valiosos em vez dos menos rentáveis.

**O Perfil de Maior Risco:** A evasão concentra-se no público maduro (45 a 55 anos), com altos saldos na conta corrente, residentes na Alemanha e do gênero feminino.

**As Causas Principais:** A perda é impulsionada por falhas de engajamento (contas inativas) e atrito gerado por excesso de produtos. Tempo de relacionamento e alta renda provaram ser ineficazes para segurar o cliente.

**Estratégia de Retenção Recomendada:** A instituição deve abandonar a política de volume ("vender mais") e focar em ativar o uso contínuo da conta. O portfólio precisa ser readequado para o segmento premium, mirando o ("sweet spot") de exatos dois produtos por cliente.

## Escolha do modelo Base: CatBoost

A seleção do **CatBoost** (*Categorical Boosting*) como algoritmo preditivo principal fundamenta-se nas características intrínsecas dos dados reveladas durante a Análise Exploratória:

* **Tratamento Nativo de Categóricas:** Variáveis críticas de negócio (como Geography e Gender) são processadas diretamente pelo algoritmo, eliminando a necessidade de *One-Hot Encoding*. Isso evita a criação de matrizes esparsas e preserva a força original da informação.
* **Captura de Padrões Não Lineares:** Como evidenciado pelo *sweet spot* de 2 produtos e pela janela de risco de idade (45-55 anos), o churn não segue uma linha reta. A estrutura de árvores de decisão do CatBoost captura essas quebras de comportamento com perfeição.
* **Prevenção de Overfitting:** Sua arquitetura interna (*Ordered Boosting*) é altamente robusta contra o sobreajuste aos dados de treino, entregando alta performance preditiva logo de saída, com mínima necessidade de tunagem inicial de hiperparâmetros.

# Dicionário de Novas Features (Visão de Negócio)

* **`Is_Balance_Zero`**: Identifica categoricamente contas sem saldo.
  * **Negócio:** Clientes com zero fundos retidos não geram receita de spread e possuem atrito mínimo para cancelar a conta. É o alerta vermelho mais claro de evasão.

* **`Active_vs_Products`**: Cruza o status de atividade com o volume de produtos contratados.
  * **Negócio:** Permite separar o cliente "fantasma" (que tem produto, mas não usa) do cliente engajado. Ajuda a focar esforços em quem realmente movimenta o ecossistema do banco.

* **`Balance_Salary_Ratio`**: Calcula qual porcentagem da renda anual do cliente está guardada no banco.
  * **Negócio:** Mede o *Share of Wallet* (fatia do bolso). Um cliente de baixa renda com alta proporção do salário no banco confia mais na instituição do que um cliente rico que deixa apenas "trocados" na conta.

* **`Age_Risk_Zone`**: Transforma a idade exata em faixas demográficas de risco.
  * **Negócio:** O comportamento de cancelamento não é linear. Facilita o direcionamento de ações do time de Marketing focando em momentos de vida (ex: Meia-Idade), não em idades exatas.


* **`Product_Danger_Zone`** Se NumOfProducts >= 3 retorna 1, senão 0.
  * **Negócio:** Isola o grupo com quase 100% de chance de churn. Evita que o modelo ache que "mais é melhor"


* **`German_Risk_Persona`**: Se Geography == 'Germany' E IsActiveMember == 0 retorna 1, senão 0.
  * **Negócio:** Aplicação sobre a conclusão do "Risco Máximo", onde a ocorrencia dessas duas variáveis gera grande risco de churn. Acelera o aprendizado das árvores de decisão, funcionando como um atalho para o aprendizado.